In [1]:
import numpy as np

np.random.seed(1)

# ---------------------------------------------------------------
# Funcion de activacion sigmoide y su derivada
# ---------------------------------------------------------------
def sig(z):
    return 1 / (1 + np.exp(-z))


def dsig(a):
    # Propiedad usada en el documento: sigma' = sigma * (1 - sigma)
    # 'a' ya es la salida de sig(z), por eso no se vuelve a aplicar sig()
    return a * (1 - a)


# ---------------------------------------------------------------
# Clase: Red neuronal feed-forward con 2 capas ocultas
# ---------------------------------------------------------------
class RedNeuronal:
    def __init__(self, n0, n1, n2, n3, eta=0.5):
        """
        n0: numero de entradas (features)
        n1: neuronas en la capa oculta 1
        n2: neuronas en la capa oculta 2
        n3: neuronas en la capa de salida
        eta: tasa de aprendizaje
        """
        self.eta = eta
        # Inicializacion aleatoria de pesos y sesgos, valores entre -1 y 1
        self.W1 = np.random.uniform(-1, 1, (n1, n0))
        self.b1 = np.random.uniform(-1, 1, (n1, 1))
        self.W2 = np.random.uniform(-1, 1, (n2, n1))
        self.b2 = np.random.uniform(-1, 1, (n2, 1))
        self.W3 = np.random.uniform(-1, 1, (n3, n2))
        self.b3 = np.random.uniform(-1, 1, (n3, 1))

    def propaga(self, X):
        """Propagacion hacia adelante. X: vector columna (n0,1)."""
        Z1 = np.dot(self.W1, X) + self.b1
        A1 = sig(Z1)
        Z2 = np.dot(self.W2, A1) + self.b2
        A2 = sig(Z2)
        Z3 = np.dot(self.W3, A2) + self.b3
        A3 = sig(Z3)
        return A3, A2, A1

    def backpropagation(self, X, Yr, A3, A2, A1):
        """Calcula el error, lo retropropaga capa por capa y ajusta pesos."""
        e = Yr - A3  # error de clasificacion en la salida

        # ---- Capa de salida (3) ----
        delta3 = e * dsig(A3)
        dW3 = np.dot(delta3, A2.T)
        db3 = delta3

        # ---- Capa oculta 2 ----
        delta2 = np.dot(self.W3.T, delta3) * dsig(A2)
        dW2 = np.dot(delta2, A1.T)
        db2 = delta2

        # ---- Capa oculta 1 ----
        delta1 = np.dot(self.W2.T, delta2) * dsig(A1)
        dW1 = np.dot(delta1, X.T)
        db1 = delta1

        # ---- Actualizacion de pesos y sesgos (descenso de gradiente) ----
        self.W3 += self.eta * dW3
        self.b3 += self.eta * db3
        self.W2 += self.eta * dW2
        self.b2 += self.eta * db2
        self.W1 += self.eta * dW1
        self.b1 += self.eta * db1

        return np.mean(np.abs(e))

    def entrenar(self, X_set, Y_set, epocas=20000, verbose=True):
        """Entrena la red con todo el conjunto de entrenamiento, epoca por epoca."""
        errores = []
        for epoca in range(epocas):
            error_epoca = 0.0
            for X, Yr in zip(X_set, Y_set):
                X = X.reshape(-1, 1)
                Yr = Yr.reshape(-1, 1)
                A3, A2, A1 = self.propaga(X)
                error_epoca += self.backpropagation(X, Yr, A3, A2, A1)
            error_promedio = error_epoca / len(X_set)
            errores.append(error_promedio)
            if verbose and epoca % 2000 == 0:
                print(f"  Epoca {epoca:6d}  Error promedio: {error_promedio:.6f}")
        return errores

    def predecir(self, X):
        X = X.reshape(-1, 1)
        A3, _, _ = self.propaga(X)
        return A3

# ---------------------------------------------------------------
# Programa principal: entrenamiento de NAND y XOR
# ---------------------------------------------------------------
if __name__ == "__main__":
    entradas = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])

    # ============= Compuerta NAND =============
    print("=" * 55)
    print("Entrenando red neuronal para la compuerta NAND")
    print("=" * 55)
    salidas_nand = np.array([[1], [1], [1], [0]])

    red_nand = RedNeuronal(n0=2, n1=4, n2=4, n3=1, eta=0.5)
    red_nand.entrenar(entradas, salidas_nand, epocas=20000)

    print("\nResultados NAND:")
    for x, y_esperado in zip(entradas, salidas_nand):
        y = red_nand.predecir(x)
        print(f"  Entrada: {x}  ->  Salida red: {y[0,0]:.4f}  "
              f"~ {round(y[0,0])}  (esperado: {y_esperado[0]})")

    # ============= Compuerta XOR =============
    print("\n" + "=" * 55)
    print("Entrenando red neuronal para la compuerta XOR")
    print("=" * 55)
    salidas_xor = np.array([[0], [1], [1], [0]])

    red_xor = RedNeuronal(n0=2, n1=4, n2=4, n3=1, eta=0.5)
    red_xor.entrenar(entradas, salidas_xor, epocas=20000)

    print("\nResultados XOR:")
    for x, y_esperado in zip(entradas, salidas_xor):
        y = red_xor.predecir(x)
        print(f"  Entrada: {x}  ->  Salida red: {y[0,0]:.4f}  "
              f"~ {round(y[0,0])}  (esperado: {y_esperado[0]})")

Entrenando red neuronal para la compuerta NAND
  Epoca      0  Error promedio: 0.411340
  Epoca   2000  Error promedio: 0.015086
  Epoca   4000  Error promedio: 0.009609
  Epoca   6000  Error promedio: 0.007515
  Epoca   8000  Error promedio: 0.006346
  Epoca  10000  Error promedio: 0.005580
  Epoca  12000  Error promedio: 0.005029
  Epoca  14000  Error promedio: 0.004609
  Epoca  16000  Error promedio: 0.004277
  Epoca  18000  Error promedio: 0.004005

Resultados NAND:
  Entrada: [0 0]  ->  Salida red: 0.9996  ~ 1  (esperado: 1)
  Entrada: [0 1]  ->  Salida red: 0.9957  ~ 1  (esperado: 1)
  Entrada: [1 0]  ->  Salida red: 0.9959  ~ 1  (esperado: 1)
  Entrada: [1 1]  ->  Salida red: 0.0063  ~ 0  (esperado: 0)

Entrenando red neuronal para la compuerta XOR
  Epoca      0  Error promedio: 0.516136
  Epoca   2000  Error promedio: 0.030135
  Epoca   4000  Error promedio: 0.014950
  Epoca   6000  Error promedio: 0.011051
  Epoca   8000  Error promedio: 0.009102
  Epoca  10000  Error promedi